[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/03_Training_Strategies/03_training_pipeline/03_training_pipeline.ipynb)

# 03. Full Training Pipeline from Scratch

**This is the most important notebook for learning HOW TO TRAIN.**

**This notebook covers:**
- Complete training loop with all best practices
- Data pipeline (loading, augmentation, batching)
- Optimizer & scheduler choices for multimodal
- Gradient accumulation (simulate large batches on small GPU)
- Mixed precision training (2x speed, half memory)
- Checkpointing and resuming
- Full training run with live visualization

---

In [ ]:
# ============================================================
#  Colab Setup (run this cell first if on Google Colab)
# ============================================================
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git"
    REPO_DIR = "/content/Multimodal-Deep-Learning"

    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
        !pip install -q -r {REPO_DIR}/requirements.txt

    os.chdir(f"{REPO_DIR}/03_Training_Strategies/03_training_pipeline")
    os.makedirs(f"{REPO_DIR}/assets", exist_ok=True)
    print(f"Colab ready — working in {os.getcwd()}")
else:
    os.makedirs("../assets", exist_ok=True)

In [ ]:
import sys
sys.path.append('../..')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.cuda.amp import autocast, GradScaler
import matplotlib.pyplot as plt
import numpy as np
import time
import os
from tqdm import tqdm
from utils.visualization import *
from utils.helpers import *

set_style()
device = get_device()

## 1. Training Recipe Overview

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Multimodal Training Pipeline', fontsize=18, fontweight='bold', pad=20)

steps = [
    (7, 9.2, 'Data Pipeline\n(images + texts + augmentation)', '#E74C3C'),
    (7, 7.8, 'Forward Pass\n(encode image → encode text → compute loss)', '#3498DB'),
    (7, 6.4, 'Backward Pass\n(compute gradients, gradient accumulation)', '#F39C12'),
    (7, 5.0, 'Optimizer Step\n(AdamW + gradient clipping)', '#2ECC71'),
    (7, 3.6, 'Scheduler Step\n(cosine decay with warmup)', '#9B59B6'),
    (7, 2.2, 'Logging & Checkpointing\n(loss, metrics, save model)', '#1ABC9C'),
    (7, 0.8, 'Evaluation\n(validation loss, retrieval accuracy)', '#34495E'),
]

for i, (x, y, label, color) in enumerate(steps):
    draw_architecture_block(ax, x, y, 9, 0.8, label, color, fontsize=10)
    if i < len(steps) - 1:
        draw_arrow(ax, (x, y - 0.5), (x, steps[i+1][1] + 0.5))

# Loop arrow
ax.annotate('', xy=(12, 9.2), xytext=(12, 0.8),
            arrowprops=dict(arrowstyle='->', color='gray', lw=2, ls='--',
                           connectionstyle='arc3,rad=0.3'))
ax.text(13, 5, 'Repeat\nfor N\nepochs', fontsize=10, color='gray', ha='center')

plt.tight_layout()
plt.savefig('../assets/training_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Data Pipeline

In [ ]:
# Create synthetic data (no download needed)
images, texts, labels = create_synthetic_image_text_pairs(
    n_samples=500, img_size=32, n_classes=5
)

# Simple tokenizer
all_words = set()
for t in texts:
    all_words.update(t.lower().split())
word2id = {w: i+2 for i, w in enumerate(sorted(all_words))}
word2id['[PAD]'] = 0
word2id['[CLS]'] = 1
vocab_size = len(word2id)

def tokenize(text, max_len=16):
    ids = [word2id['[CLS]']] + [word2id.get(w, 0) for w in text.lower().split()]
    ids = ids[:max_len]
    ids += [0] * (max_len - len(ids))
    return torch.tensor(ids)

# Data augmentation for images
import torchvision.transforms as T

train_transform = T.Compose([
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.RandomErasing(p=0.1),
])

class MultimodalDataset(Dataset):
    def __init__(self, images, texts, transform=None):
        self.images = images
        self.texts = texts
        self.transform = transform
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img = self.images[idx]
        if self.transform:
            img = self.transform(img)
        txt = tokenize(self.texts[idx])
        return img, txt

# Train/val split
n_train = int(0.8 * len(images))
train_dataset = MultimodalDataset(images[:n_train], texts[:n_train], train_transform)
val_dataset = MultimodalDataset(images[n_train:], texts[n_train:])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"Train: {len(train_dataset)} samples, {len(train_loader)} batches")
print(f"Val:   {len(val_dataset)} samples, {len(val_loader)} batches")
print(f"Vocab: {vocab_size} words")

## 3. Model + Optimizer + Scheduler

In [ ]:
# Reuse CLIP model from Module 02

class CLIPModel(nn.Module):
    def __init__(self, embed_dim=128, proj_dim=64, vocab_size=100):
        super().__init__()
        # Image encoder
        self.img_patch = nn.Conv2d(3, embed_dim, 4, 4)
        self.img_cls = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        self.img_pos = nn.Parameter(torch.randn(1, 65, embed_dim) * 0.02)
        img_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=4, dim_feedforward=256, batch_first=True)
        self.img_transformer = nn.TransformerEncoder(img_layer, num_layers=3)
        self.img_norm = nn.LayerNorm(embed_dim)
        self.img_proj = nn.Linear(embed_dim, proj_dim)

        # Text encoder
        self.tok_embed = nn.Embedding(vocab_size, embed_dim)
        self.txt_pos = nn.Embedding(64, embed_dim)
        txt_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=4, dim_feedforward=256, batch_first=True)
        self.txt_transformer = nn.TransformerEncoder(txt_layer, num_layers=3)
        self.txt_norm = nn.LayerNorm(embed_dim)
        self.txt_proj = nn.Linear(embed_dim, proj_dim)

        self.logit_scale = nn.Parameter(torch.ones(1) * np.log(1 / 0.07))

    def encode_image(self, x):
        B = x.shape[0]
        x = self.img_patch(x).flatten(2).transpose(1, 2)
        x = torch.cat([self.img_cls.expand(B, -1, -1), x], dim=1)
        x = x + self.img_pos
        x = self.img_transformer(x)
        x = self.img_norm(x[:, 0])
        return F.normalize(self.img_proj(x), dim=-1)

    def encode_text(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0).expand(B, -1)
        x = self.tok_embed(x) + self.txt_pos(pos)
        x = self.txt_transformer(x)
        x = self.txt_norm(x[:, 0])
        return F.normalize(self.txt_proj(x), dim=-1)

    def forward(self, images, text_ids):
        img_emb = self.encode_image(images)
        txt_emb = self.encode_text(text_ids)
        logit_scale = self.logit_scale.exp()
        return logit_scale * img_emb @ txt_emb.T


model = CLIPModel(embed_dim=128, proj_dim=64, vocab_size=vocab_size).to(device)
count_parameters(model)

In [ ]:
# Optimizer: AdamW (always use this for transformers)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

# Scheduler: Cosine decay with linear warmup
total_steps = len(train_loader) * 50  # 50 epochs
warmup_steps = total_steps // 10      # 10% warmup

def get_lr(step):
    if step < warmup_steps:
        return step / warmup_steps
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return 0.5 * (1 + np.cos(np.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, get_lr)

# Visualize learning rate schedule
lrs = [get_lr(s) * 3e-4 for s in range(total_steps)]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(lrs, color='#E74C3C', linewidth=2)
ax.axvline(x=warmup_steps, color='gray', linestyle='--', label='End of warmup')
ax.set_xlabel('Training Step')
ax.set_ylabel('Learning Rate')
ax.set_title('Cosine Decay with Linear Warmup', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Total steps: {total_steps}")
print(f"Warmup steps: {warmup_steps}")
print(f"Peak LR: 3e-4")

## 4. Gradient Accumulation (Essential for Low Compute!)

**Problem:** CLIP needs large batch sizes (1000+) but you only have memory for 32.  
**Solution:** Accumulate gradients over multiple mini-batches before stepping.

In [ ]:
# Visualize gradient accumulation
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Without accumulation
ax = axes[0]
ax.set_title('Without Gradient Accumulation\n(batch=32, effective=32)', fontsize=12, fontweight='bold')
steps_labels = ['Batch 1\n(32)', 'Step', 'Batch 2\n(32)', 'Step', 'Batch 3\n(32)', 'Step']
colors = ['#3498DB', '#E74C3C', '#3498DB', '#E74C3C', '#3498DB', '#E74C3C']
ax.barh(range(6), [1]*6, color=colors, alpha=0.7)
for i, label in enumerate(steps_labels):
    ax.text(0.5, i, label, ha='center', va='center', fontsize=9, fontweight='bold')
ax.set_xlim(0, 1)
ax.axis('off')

# With accumulation (4 steps)
ax = axes[1]
ax.set_title('With Gradient Accumulation (4 steps)\n(batch=32, effective=128)', 
             fontsize=12, fontweight='bold', color='#2ECC71')
steps_labels = ['Batch 1 (32)', 'Batch 2 (32)', 'Batch 3 (32)', 'Batch 4 (32)', 
                'STEP\n(accumulated)', 'Batch 5...']
colors = ['#3498DB', '#3498DB', '#3498DB', '#3498DB', '#E74C3C', '#3498DB']
bars = ax.barh(range(6), [1]*6, color=colors, alpha=0.7)
for i, label in enumerate(steps_labels):
    ax.text(0.5, i, label, ha='center', va='center', fontsize=9, fontweight='bold')
ax.set_xlim(0, 1)
ax.axis('off')

plt.tight_layout()
plt.savefig('../assets/gradient_accumulation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Full training loop with all best practices

def train_multimodal(
    model, train_loader, val_loader, optimizer, scheduler,
    n_epochs=50, grad_accum_steps=4, max_grad_norm=1.0,
    device='cpu'
):
    """Production-ready training loop for multimodal models."""
    history = {'train_loss': [], 'val_loss': [], 'lr': [], 'epoch_time': []}
    best_val_loss = float('inf')
    global_step = 0

    for epoch in range(n_epochs):
        start_time = time.time()
        model.train()
        epoch_loss = 0
        n_batches = 0

        for batch_idx, (images, text_ids) in enumerate(train_loader):
            images = images.to(device)
            text_ids = text_ids.to(device)

            # Forward pass
            logits = model(images, text_ids)
            labels = torch.arange(len(images), device=device)
            loss = (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2

            # Scale loss for gradient accumulation
            loss = loss / grad_accum_steps
            loss.backward()

            if (batch_idx + 1) % grad_accum_steps == 0:
                # Gradient clipping
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                
                # Optimizer step
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1

            epoch_loss += loss.item() * grad_accum_steps
            n_batches += 1

        avg_train_loss = epoch_loss / n_batches
        history['train_loss'].append(avg_train_loss)
        history['lr'].append(optimizer.param_groups[0]['lr'])

        # Validation
        model.eval()
        val_loss = 0
        n_val = 0
        with torch.no_grad():
            for images, text_ids in val_loader:
                images = images.to(device)
                text_ids = text_ids.to(device)
                logits = model(images, text_ids)
                labels = torch.arange(len(images), device=device)
                loss = (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2
                val_loss += loss.item()
                n_val += 1

        avg_val_loss = val_loss / max(n_val, 1)
        history['val_loss'].append(avg_val_loss)

        elapsed = time.time() - start_time
        history['epoch_time'].append(elapsed)

        # Checkpointing (save best model)
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': best_val_loss,
            }, '../assets/best_model.pt')

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{n_epochs} | "
                  f"Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f} | "
                  f"LR: {optimizer.param_groups[0]['lr']:.6f} | Time: {elapsed:.1f}s")

    return history


# Reset model and optimizer
model = CLIPModel(embed_dim=128, proj_dim=64, vocab_size=vocab_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
total_steps = len(train_loader) * 50 // 4  # account for grad accumulation
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)

# Train!
print("Training with gradient accumulation (effective batch = 32 × 4 = 128)...")
history = train_multimodal(
    model, train_loader, val_loader, optimizer, scheduler,
    n_epochs=50, grad_accum_steps=4, device=device
)

In [ ]:
# Visualize training results
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Training Results', fontsize=16, fontweight='bold')

# Loss curves
ax = axes[0]
ax.plot(history['train_loss'], label='Train', color='#E74C3C', linewidth=2)
ax.plot(history['val_loss'], label='Validation', color='#3498DB', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Loss Curves')
ax.legend()

# Learning rate
ax = axes[1]
ax.plot(history['lr'], color='#2ECC71', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_title('Learning Rate Schedule')

# Epoch time
ax = axes[2]
ax.bar(range(len(history['epoch_time'])), history['epoch_time'], color='#F39C12', alpha=0.7)
ax.set_xlabel('Epoch')
ax.set_ylabel('Time (seconds)')
ax.set_title('Epoch Duration')

plt.tight_layout()
plt.savefig('../assets/training_results.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nTotal training time: {sum(history['epoch_time']):.1f}s")
print(f"Best validation loss: {min(history['val_loss']):.4f}")

## Training Checklist for Low Compute

| Technique | Memory Saving | Speed | Difficulty |
|-----------|--------------|-------|------------|
| **Gradient Accumulation** | None (same mem) | Same | Easy |
| **Mixed Precision (fp16)** | ~50% | ~2x | Easy |
| **Gradient Checkpointing** | ~60% | ~0.8x | Medium |
| **Freeze Encoders** | ~50%+ | ~2x | Easy |
| **LoRA/QLoRA** | ~75-90% | ~1.5x | Medium |
| **Smaller Model** | Proportional | Proportional | Easy |

---
**Next:** Module 04 - Finetuning for Low Compute (LoRA, QLoRA, Adapters!)

---

## Real-World Multi-Stage Training Pipelines

> **Everything above teaches you the TOOLS.** This section teaches you how production teams **combine** those tools into a complete pipeline that turns a raw base model into a deployed, aligned, multi-task multimodal system.

Production VLMs (LLaVA, InternVL, Qwen-VL, DeepSeek-VL) are **never** trained in a single stage. The modern recipe is a **3-stage pipeline**:

### The Universal 3-Stage Pattern

```
┌──────────────────────────────────────────────────────────────────────────────────┐
│                         THE 3-STAGE TRAINING PIPELINE                            │
│                                                                                  │
│                          ┌─────────────────────┐                                 │
│                          │   RAW BASE MODEL     │                                 │
│                          │ (random / pretrained) │                                 │
│                          └─────────┬───────────┘                                 │
│                                    │                                              │
│      ┌─────────────────────────────▼──────────────────────────────┐              │
│      │  STAGE 1: SUPERVISED FINE-TUNING (SFT)                     │              │
│      │  • Next-token prediction on curated (image, text) pairs     │              │
│      │  • Optional knowledge distillation from teacher model       │              │
│      │  • Math: L = -Σₜ log P(yₜ | y₁…yₜ₋₁, I)                  │              │
│      │  • Duration: days to weeks on 8-64 GPUs                     │              │
│      └──────────────────────┬─────────────────────────────────────┘              │
│                             │                                                     │
│                             ▼                                                     │
│      ┌──────────────────────────────────────────────────────────────┐             │
│      │  STAGE 2: REINFORCEMENT LEARNING (RLHF / DPO / GRPO)        │             │
│      │  • Optimize a reward signal beyond what SFT can capture      │             │
│      │  • Choose ONE algorithm per run:                              │             │
│      │    RLHF (PPO + reward model)                                  │             │
│      │    DPO  (direct preference, no reward model)                  │             │
│      │    GRPO (group-relative, verifiable rewards)                  │             │
│      │  • Produces multiple specialized checkpoints                  │             │
│      └──────────────────────┬───────────────────────────────────────┘             │
│                             │                                                     │
│                             ▼                                                     │
│      ┌──────────────────────────────────────────────────────────────┐             │
│      │  STAGE 3: WEIGHT-SPACE MERGING / MODEL SOUPING               │             │
│      │  • Combine checkpoints in weight space (no training!)         │             │
│      │  • Methods: Linear soup, Task arithmetic, TIES, DARE          │             │
│      │  • Cost: minutes on CPU                                       │             │
│      └──────────────────────┬───────────────────────────────────────┘             │
│                             │                                                     │
│                             ▼                                                     │
│                    ┌───────────────────┐                                          │
│                    │   FINAL MODEL     │                                          │
│                    │  (deploy / serve) │                                          │
│                    └───────────────────┘                                          │
└──────────────────────────────────────────────────────────────────────────────────┘
```

### Stage 1: Supervised Fine-Tuning (SFT) — In Depth

SFT trains the model on curated input-output pairs using the **cross-entropy objective**. For a multimodal model with image $I$ and response tokens $y_1, \ldots, y_T$:

$$\mathcal{L}_{\text{SFT}}(\theta) = -\sum_{t=1}^{T} \log P_\theta(y_t \mid y_1, \ldots, y_{t-1}, I)$$

At each timestep, the model produces a logit vector $\mathbf{z}_t \in \mathbb{R}^V$ over vocabulary size $V$:

$$P_\theta(y_t = w \mid \cdot) = \frac{e^{z_{t,w}}}{\sum_{w'=1}^{V} e^{z_{t,w'}}}, \qquad \nabla_{z_{t,w}} \mathcal{L} = P_\theta(w \mid \cdot) - \mathbb{1}[w = y_t]$$

**Key:** Only compute loss on **response tokens**, not instruction/image tokens:

```
Tokens:  <img₁> ... <img₅₇₆> [INST] What is this? [/INST] This is a cat .
Loss:     ✗          ✗        ✗      ✗     ✗  ✗     ✗     ✓    ✓  ✓ ✓  ✓
```

#### Knowledge Distillation in SFT

Train a smaller student $S$ to mimic a larger teacher $T$:

$$\mathcal{L}_{\text{SFT+KD}} = (1-\alpha)\,\mathcal{L}_{\text{CE}}(S(x), y) + \alpha\, T_{\text{temp}}^2 \,\text{KL}\!\left(\text{softmax}\!\left(\frac{z_S}{T_{\text{temp}}}\right) \;\Big\|\; \text{softmax}\!\left(\frac{z_T}{T_{\text{temp}}}\right)\right)$$

**Why $T^2$ scaling?** The KL gradient is $\frac{1}{T}(p_i^S - p_i^T)$ — shrinks as $T$ increases. Multiplying by $T^2$ compensates.

```
  Teacher (235B, frozen)                               Student (1-7B, training)
  ┌──────────────────────┐                             ┌──────────────────────┐
  │   Qwen3-VL-235B      │                             │   Target model       │
  │                      │                             │                      │
  │  Image + Text ───► z_T ──── softmax(z_T / T) ─────►│ KL divergence        │
  │                      │       "soft targets"        │    ▲                  │
  └──────────────────────┘       P_T(y|x)              │    │                  │
                                                        │  z_S ◄── Image+Text  │
  Ground-truth labels y* ──────────────────────────────►│ CE loss              │
                                                        │                      │
                                                        │  Total loss:         │
                                                        │  L = (1-α)·CE(S,y*) │
                                                        │    + α·T²·KL(S║T)   │
                                                        └──────────────────────┘
```

#### SFT Hyperparameter Table

| Component | Typical Choice | Why |
|-----------|---------------|-----|
| Optimizer | AdamW ($\beta_1$=0.9, $\beta_2$=0.95) | Stable for transformers |
| Weight decay | $\lambda$ = 0.1 | Prevents overfitting |
| Peak LR | 1e-5 to 2e-5 | Fine-tuning regime |
| Warmup | 1–3% of total steps | Stabilizes early Adam |
| Schedule | Cosine decay | Smooth annealing |
| Precision | bf16 + FlashAttention-2 | 50% memory savings |
| Batch size | 256–1024 (grad accumulation) | Stable gradients |
| Gradient clipping | max_norm = 1.0 | Prevents spikes |
| Epochs | 1–3 | Overfitting is fast |

### Stage 2: Reinforcement Learning — RLHF, DPO, GRPO

After SFT, the model can follow instructions but may still hallucinate or produce formatting errors. **RL refines behavior** by optimizing a reward signal beyond what maximum likelihood can capture.

```
Why RL after SFT?

SFT optimizes:     max P(y* | x)         ← "match the reference answer exactly"
RL  optimizes:     max E[R(x, y)]        ← "produce ANY answer that scores well"

SFT limitation: Many valid outputs, but SFT only sees ONE reference.
RL advantage:   Can optimize non-differentiable metrics (CIDEr, IoU, edit distance)
```

---

#### 2a. RLHF (Reinforcement Learning from Human Feedback)

**Step 1: Train a reward model** $R_\phi$ on human preference pairs $(y_w \succ y_l)$ using the Bradley-Terry model:

$$P(y_w \succ y_l \mid x) = \sigma\!\left(R_\phi(x, y_w) - R_\phi(x, y_l)\right)$$

$$\mathcal{L}_{\text{RM}}(\phi) = -\mathbb{E}_{(x, y_w, y_l)} \left[\log \sigma\!\left(R_\phi(x, y_w) - R_\phi(x, y_l)\right)\right]$$

**Step 2: Optimize the policy** $\pi_\theta$ using PPO with KL regularization:

$$\max_{\pi_\theta} \;\mathbb{E}_{x, y \sim \pi_\theta} \left[R_\phi(x, y)\right] - \beta\, \text{KL}\!\left(\pi_\theta \;\|\; \pi_{\text{ref}}\right)$$

**The PPO Clipped Surrogate Objective:**

$$\mathcal{L}_{\text{PPO}}^{\text{CLIP}} = -\mathbb{E}_t \left[\min\!\left(r_t(\theta)\,\hat{A}_t, \;\text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon)\,\hat{A}_t\right)\right]$$

where $r_t(\theta) = \frac{\pi_\theta(a_t \mid s_t)}{\pi_{\theta_{\text{old}}}(a_t \mid s_t)}$ and $\hat{A}_t$ is the advantage estimated via GAE:

$$\hat{A}_t^{\text{GAE}} = \sum_{l=0}^{\infty} (\gamma \lambda)^l \delta_{t+l}, \quad \delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$

```
  PPO Clipping Mechanism (ε = 0.2):

  When Â > 0 (good action):           When Â < 0 (bad action):
  ┌─────────────────────────┐         ┌─────────────────────────┐
  │  Objective              │         │  Objective              │
  │    ^         ┌────────  │         │    ^                    │
  │    │        /│          │         │    │  ────────┐         │
  │    │       / │          │         │    │          │\        │
  │    │      /  │          │         │    │          │ \       │
  │    │─────/───┼────► r   │         │    │──────────┼──\► r   │
  │       0.8  1.0  1.2     │         │       0.8  1.0  1.2    │
  │  Cap at r=1+ε           │         │  Cap at r=1-ε          │
  │  (prevent over-exploit) │         │  (prevent over-correct)│
  └─────────────────────────┘         └─────────────────────────┘
```

**RLHF Memory: 4 models simultaneously → ~78 GB for 7B:**

| Model | Role | Memory |
|-------|------|--------|
| Policy $\pi_\theta$ (trainable) | Generates responses | 14 GB |
| Reference $\pi_{\text{ref}}$ (frozen) | KL anchor | 14 GB |
| Reward $R_\phi$ (frozen) | Scores responses | 14 GB |
| Value $V_\psi$ (trainable) | Estimates advantage | 14 GB |
| Optimizer + activations | | ~22 GB |
| **Total** | | **~78 GB** |

#### 2b. DPO (Direct Preference Optimization) — Full Derivation

DPO eliminates the reward model entirely. The key insight: the optimal policy under KL-constrained reward maximization has a **closed-form solution**.

**Start:** The RLHF objective:

$$\max_{\pi} \;\mathbb{E}_{y \sim \pi}\left[r(x,y)\right] - \beta\, \text{KL}(\pi \| \pi_{\text{ref}})$$

**Solve for optimal policy** (via calculus of variations, setting $\frac{\partial J}{\partial \pi} = 0$):

$$\pi^*(y \mid x) = \frac{1}{Z(x)} \pi_{\text{ref}}(y \mid x) \exp\!\left(\frac{r(x,y)}{\beta}\right)$$

**Invert for reward:**

$$r(x, y) = \beta \log \frac{\pi^*(y \mid x)}{\pi_{\text{ref}}(y \mid x)} + \beta \log Z(x)$$

**Substitute into Bradley-Terry** — the $Z(x)$ cancels (same prompt for $y_w$ and $y_l$):

$$\boxed{\mathcal{L}_{\text{DPO}}(\theta) = -\mathbb{E}_{(x, y_w, y_l)}\left[\log \sigma\!\left(\beta \log \frac{\pi_\theta(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \beta \log \frac{\pi_\theta(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)}\right)\right]}$$

**Implicit reward:** $r(x, y) = \beta \log \frac{\pi_\theta(y \mid x)}{\pi_{\text{ref}}(y \mid x)}$ — no reward model needed!

**DPO Gradient:**

$$\nabla_\theta \mathcal{L}_{\text{DPO}} = -\beta\, \mathbb{E}\left[\underbrace{\sigma(\hat{r}_l - \hat{r}_w)}_{\text{weight (large when wrong)}}\left[\nabla \log \pi_\theta(y_w) - \nabla \log \pi_\theta(y_l)\right]\right]$$

When the model incorrectly ranks the pair ($\hat{r}_l > \hat{r}_w$), the weight $\sigma(\hat{r}_l - \hat{r}_w) \approx 1$ → large gradient. When correctly ranked → small gradient. **DPO automatically focuses on hard pairs.**

**DPO Memory: Only 2 models → ~38 GB (or ~18 GB with LoRA):**

```
┌──────────────────────────────────────────────┐
│  DPO Memory (7B model)                        │
├──────────────────────────────────────────────┤
│  Policy π_θ (trainable)     : 14 GB (bf16)   │
│  Reference π_ref (frozen)   : 14 GB (bf16)   │
│  Optimizer + activations    : ~10 GB          │
│  TOTAL                      : ~38 GB → 1×A100│
│                                               │
│  With LoRA: π_ref = frozen base (FREE!)       │
│  TOTAL with LoRA            : ~18 GB → RTX4090│
└──────────────────────────────────────────────┘
```

#### 2c. GRPO (Group Relative Policy Optimization) — Full Algorithm

GRPO (DeepSeek-R1, 2024) eliminates both the reward model AND the value model by using **group-relative advantages** from verifiable rewards.

**Step 1 — Generate group of G responses per prompt:**

$$\lbrace y_1, y_2, \ldots, y_G \rbrace \sim \pi_{\theta_{\text{old}}}(\cdot \mid x)$$

**Step 2 — Score with verifiable reward and normalize within group:**

$$A_i = \frac{R(x, y_i) - \mu_G}{\sigma_G + \epsilon}, \quad \mu_G = \frac{1}{G}\sum_{j=1}^{G} R_j, \quad \sigma_G = \sqrt{\frac{1}{G}\sum_{j=1}^{G}(R_j - \mu_G)^2}$$

**Step 3 — Clipped policy gradient with KL regularization:**

$$\mathcal{L}_{\text{GRPO}} = -\frac{1}{G}\sum_{i=1}^{G} \min\!\left(r_i A_i,\;\text{clip}(r_i, 1-\epsilon, 1+\epsilon)\, A_i\right) + \beta\,\text{KL}(\pi_\theta \| \pi_{\text{ref}})$$

where $r_i = \frac{\pi_\theta(y_i \mid x)}{\pi_{\theta_{\text{old}}}(y_i \mid x)}$ is the importance sampling ratio.

**Verifiable Reward Functions (RLVR):**

| Task | Reward $R(x, y)$ | Auto-gradeable? |
|------|-----------------|----------------|
| OCR | $1 - \text{CER}(y, y^*)$ | ✓ |
| Math | $\mathbb{1}[\text{extract}(y) = a^*]$ | ✓ |
| Bounding box | $\text{IoU}(\text{bbox}(y), \text{bbox}^*)$ | ✓ |
| Code | tests_passed / total_tests | ✓ |
| Format | $\mathbb{1}[\text{matches\_schema}(y)]$ | ✓ |

**GRPO Step-by-Step Numerical Example** ($G=4$, $\epsilon=0.2$):

Prompt: "OCR this receipt image"

| Response | Predicted text | $R_i$ |
|----------|---------------|-------|
| $y_1$ | "Total: \$42.50" (perfect) | **1.10** |
| $y_2$ | "Total: \$42.5O" (one char wrong) | **1.03** |
| $y_3$ | "Totla: \$42.50" (typo) | **1.03** |
| $y_4$ | "Total 42.50 total 42.50" (repetition) | **0.40** |

$\mu = 0.89$, $\sigma = 0.276$

| | $R_i - \mu$ | $A_i$ | Action |
|--|------------|-------|--------|
| $y_1$ | +0.21 | **+0.76** | Reinforce ↑ |
| $y_2$ | +0.14 | **+0.51** | Mild reinforce ↑ |
| $y_3$ | +0.14 | **+0.51** | Mild reinforce ↑ |
| $y_4$ | -0.49 | **-1.78** | Strongly suppress ↓ |

**GRPO Training Loop — Complete Diagram:**

```
┌──────────────────────────────────────────────────────────────────────────────┐
│                          GRPO TRAINING LOOP                                  │
│                                                                              │
│  for each batch of prompts {x₁, x₂, ..., x_B}:                            │
│                                                                              │
│  ┌────────────────────────────────────────────────────────────────────────┐  │
│  │  STEP 1: ROLLOUT GENERATION (via vLLM)                                │  │
│  │  for each xₖ: generate G responses {y₁..y_G} ~ π_θ_old(·|xₖ)        │  │
│  │  Store log-probs: log π_θ_old(yᵢ|xₖ)                                 │  │
│  └──────────────────────────────┬─────────────────────────────────────────┘  │
│                                 ▼                                            │
│  ┌────────────────────────────────────────────────────────────────────────┐  │
│  │  STEP 2: REWARD SCORING (no gradient, parallelizable)                 │  │
│  │  Rᵢ = w₁·R_task(xₖ,yᵢ) + w₂·R_format(xₖ,yᵢ) + w₃·R_rep(xₖ,yᵢ)    │  │
│  └──────────────────────────────┬─────────────────────────────────────────┘  │
│                                 ▼                                            │
│  ┌────────────────────────────────────────────────────────────────────────┐  │
│  │  STEP 3: ADVANTAGE (per-group normalization)                          │  │
│  │  for each xₖ: Aᵢ = (Rᵢ - mean(R)) / (std(R) + ε)                   │  │
│  └──────────────────────────────┬─────────────────────────────────────────┘  │
│                                 ▼                                            │
│  ┌────────────────────────────────────────────────────────────────────────┐  │
│  │  STEP 4: POLICY GRADIENT UPDATE (with gradient)                       │  │
│  │  rᵢ = exp(log π_θ(yᵢ|x) - log π_θ_old(yᵢ|x))                       │  │
│  │  L = -min(rᵢ·Aᵢ, clip(rᵢ,1±ε)·Aᵢ) + β·KL(π_θ‖π_ref)              │  │
│  │  Backward + AdamW step                                                │  │
│  └──────────────────────────────┬─────────────────────────────────────────┘  │
│                                 ▼                                            │
│  ┌────────────────────────────────────────────────────────────────────────┐  │
│  │  STEP 5: SYNC — θ_old ← θ                                            │  │
│  └────────────────────────────────────────────────────────────────────────┘  │
│                                                                              │
│  Repeat until convergence (1-3 epochs over prompt dataset)                  │
└──────────────────────────────────────────────────────────────────────────────┘
```

#### RL Algorithm Comparison — Complete Analysis

| Property | **RLHF (PPO)** | **DPO** | **GRPO** |
|----------|----------------|---------|----------|
| **Reward model** | Required (learned $R_\phi$) | Not needed | Not needed (verifiable) |
| **Value model** | Required (learned $V_\psi$) | Not needed | Not needed |
| **Models in memory** | 4 (π, π_ref, R, V) | 2 (π, π_ref) | 2 (π, π_ref) |
| **Memory (7B, bf16)** | ~78 GB | ~38 GB | ~40 GB |
| **Stability** | Moderate | High | High |
| **Training data** | Preference pairs | Preference pairs | Prompts + reward fn |
| **Human labels needed** | Yes | Yes | No |
| **Reward hacking risk** | Moderate | Low | Low (verifiable) |
| **Online/Offline** | Online | Offline | Online |
| **Best for** | General alignment | Preference tuning | Tasks with auto-grading |
| **Key paper** | InstructGPT (2022) | Rafailov et al. (2023) | DeepSeek-R1 (2024) |

**Decision flowchart:**

```
  Do you have human preference data?
       │
       ├── YES ──► Is it expensive to collect more?
       │               ├── YES ──► DPO (offline, efficient with existing data)
       │               └── NO  ──► RLHF/PPO (online, explores with reward model)
       │
       └── NO  ──► Can you auto-grade responses?
                       ├── YES ──► GRPO (verifiable rewards, no humans needed)
                       └── NO  ──► Collect human labels first, then DPO
```

### Stage 3: Weight-Space Merging (Model Souping)

After training multiple specialized models via RL, **weight merging** combines their strengths in weight space — **no additional training, minutes on CPU**.

#### Why Merging Works

Fine-tuned models from the same base lie in the **same loss basin**. Linear paths between them stay in low-loss regions — merging finds a point that's good for ALL tasks.

---

#### Method 1: Linear Interpolation (Model Soup)

Weighted average of full checkpoints:

$$\theta_{\text{merged}} = \sum_{k=1}^{K} \alpha_k \, \theta_k, \qquad \sum_{k=1}^{K} \alpha_k = 1$$

**Greedy soup:** Start with the best model, greedily add others only if they improve validation accuracy.

---

#### Method 2: Task Arithmetic

Define a **task vector** $\tau_k = \theta_k^{\text{fine}} - \theta^{\text{base}}$ (the delta from base to fine-tuned):

$$\theta_{\text{merged}} = \theta^{\text{base}} + \sum_{k=1}^{K} \lambda_k \, \tau_k$$

Three operations:
- **Addition:** $\theta = \theta^{\text{base}} + \lambda_1 \tau_{\text{OCR}} + \lambda_2 \tau_{\text{bbox}}$ → gains BOTH skills
- **Negation:** $\theta = \theta^{\text{base}} - 0.5 \cdot \tau_{\text{toxic}}$ → REMOVES toxicity
- **Analogy:** Transfer skill deltas between domains

**Works when task vectors are nearly orthogonal** ($\tau_j^\top H \tau_k \approx 0$) — each independently reduces its task's loss.

---

#### Method 3: TIES Merging (Trim, Elect Sign, Disjoint Merge)

Addresses **interference** when task vectors modify the same parameter in opposite directions:

**Step 1 — Trim:** Zero out small-magnitude entries (keep top-$p\%$):

$$\tilde{\tau}_{k}^{(j)} = \begin{cases} \tau_k^{(j)} & \text{if } |\tau_k^{(j)}| \geq \text{quantile}(|\tau_k|, 1-p) \\ 0 & \text{otherwise} \end{cases}$$

**Step 2 — Elect sign:** Majority vote per parameter:

$$s^{(j)} = \text{sign}\!\left(\sum_{k} \tilde{\tau}_k^{(j)}\right)$$

**Step 3 — Disjoint merge:** Average only values matching elected sign:

$$\theta_{\text{merged}}^{(j)} = \theta_{\text{base}}^{(j)} + \lambda \cdot \frac{1}{|\mathcal{A}_j|}\sum_{k \in \mathcal{A}_j} \tilde{\tau}_k^{(j)}, \quad \mathcal{A}_j = \lbrace k : \text{sign}(\tilde{\tau}_k^{(j)}) = s^{(j)} \rbrace$$

**TIES Worked Example** (3 task vectors, 6 params, keep top 50%):

```
Raw vectors:
τ_A = [+0.8, -0.3, +0.1, -0.7, +0.02, +0.5]
τ_B = [+0.6, +0.4, -0.2, -0.5, +0.01, -0.3]
τ_C = [-0.1, -0.2, +0.9, +0.3, -0.6,  +0.4]

After TRIM (keep top 3 per vector):
τ̃_A = [+0.8,  0,    0,   -0.7,  0,   +0.5]
τ̃_B = [+0.6, +0.4,  0,   -0.5,  0,    0  ]
τ̃_C = [ 0,    0,   +0.9,  0,   -0.6, +0.4]

ELECT SIGN (sum → majority):
j=1: +0.8+0.6  = +1.4 → s=+    j=4: -0.7-0.5    = -1.2 → s=−
j=2: +0.4      = +0.4 → s=+    j=5: -0.6        = -0.6 → s=−
j=3: +0.9      = +0.9 → s=+    j=6: +0.5+0.4    = +0.9 → s=+

DISJOINT MERGE (avg matching-sign values):
merged = [+0.70, +0.40, +0.90, -0.60, -0.60, +0.45]

θ_merged = θ_base + λ · merged
```

#### Merging Method Comparison

| Method | Interference handling | Quality |
|--------|----------------------|---------|
| **Linear (Soup)** | None (naive avg) | Good |
| **Task Arithmetic** | None | Good |
| **TIES** | Trim + sign election | Better |
| **DARE + TIES** | Dropout + trim + sign | **Best** |

```
  Full Merging Visualization:

  θ_base ─── SFT ─── θ_SFT
                        │
                        ├── DPO (safety)  → τ_safe
                        ├── DPO (quality) → τ_helpful
                        ├── GRPO (OCR)    → τ_ocr
                        └── GRPO (bbox)   → τ_bbox
                                    │
                              Merge: θ = θ_SFT + Σ λₖτₖ
                                    │
                              θ_merged (ALL capabilities)
                              Cost: minutes, no GPU needed
```

### End-to-End Example 1: LLaVA-Style 3-Stage Pipeline

```
┌──────────────────────────────────────────────────────────────────────────────┐
│                  LLaVA-Style Full Training Pipeline                          │
├──────────────────────────────────────────────────────────────────────────────┤
│                                                                              │
│  Stage 1: SFT — Phase 1 (Alignment)                                         │
│  ═════════════════════════════════════                                        │
│  Dataset:   595K image-caption pairs (CC3M filtered)                         │
│  Frozen:    CLIP ViT-L/14 + Vicuna-7B                                       │
│  Trainable: MLP projector only (~21M params)                                 │
│  Objective: L = -Σₜ log P(yₜ | y₁…yₜ₋₁, MLP(ViT(I)))                      │
│  Config:    AdamW, lr=1e-3, cosine, bf16, batch=256                         │
│  Duration:  ~24h on 8×A100                                                   │
│                                                                              │
│  ┌────────────┐   ┌──────────┐   ┌──────────────┐   ┌────────────┐         │
│  │  Image I   │──►│ CLIP ViT │──►│ MLP Projector│──►│ Vicuna-7B  │         │
│  └────────────┘   │ (frozen) │   │ (TRAINING)   │   │ (frozen)   │         │
│                   └──────────┘   └──────────────┘   └────────────┘         │
│  Output: → LLaVA-base                                                       │
│                   │                                                          │
│                   ▼                                                          │
│  Stage 1.5: SFT — Phase 2 (Instruction Tuning)                              │
│  ═══════════════════════════════════════════════                              │
│  Dataset:   150K instruction-following conversations                         │
│  Frozen:    CLIP ViT-L/14 only                                               │
│  Trainable: MLP projector + LoRA on LLM (r=128, α=256)                     │
│  Config:    AdamW, lr=2e-5, cosine, bf16, batch=128                         │
│  Output: → LLaVA-v1.5-7B                                                    │
│                   │                                                          │
│                   ▼                                                          │
│  Stage 2: DPO Alignment                                                      │
│  ═══════════════════════                                                      │
│  Algorithm: DPO (β = 0.1), no reward model                                  │
│  Data:      10K preference pairs (y_w, y_l)                                  │
│  Trainable: LoRA on LLM (r=64) + projector                                  │
│  Config:    AdamW, lr=5e-7, bf16, batch=32                                   │
│                                                                              │
│  Train TWO specialized variants:                                             │
│                                                                              │
│  ┌───────────────────────┐       ┌───────────────────────┐                  │
│  │ Variant A: Safety DPO │       │ Variant B: Quality DPO│                  │
│  │ y_w: refuses harmful  │       │ y_w: detailed, grounded│                 │
│  │ y_l: complies unsafely│       │ y_l: vague, hallucin. │                  │
│  └──────────┬────────────┘       └──────────┬────────────┘                  │
│             │                               │                                │
│             ▼                               ▼                                │
│  → θ_safe (τ_safe)               → θ_helpful (τ_helpful)                    │
│             │                               │                                │
│             └──────────────┬────────────────┘                                │
│                            ▼                                                 │
│  Stage 3: Weight Merging                                                     │
│  ═══════════════════════════                                                  │
│  Method:  Task arithmetic                                                    │
│  Formula: θ = θ_SFT + 0.7·τ_safe + 0.5·τ_helpful                           │
│  Tuning:  Grid search λ ∈ {0.1, 0.3, 0.5, 0.7, 0.9}                       │
│  Cost:    ~5 minutes on CPU                                                  │
│                                                                              │
│  ┌────────────────────┬──────┬───────────┬───────────┬────────┐             │
│  │ Model              │ MMMU │ POPE (↓H) │ LLaVA-Bench│Safety │             │
│  ├────────────────────┼──────┼───────────┼───────────┼────────┤             │
│  │ LLaVA-SFT          │ 35.2 │ 82.1      │ 68.4      │ 72.0   │             │
│  │ LLaVA-safe         │ 33.8 │ 88.5      │ 64.1      │ 91.2   │             │
│  │ LLaVA-helpful      │ 36.1 │ 83.0      │ 74.8      │ 73.5   │             │
│  │ LLaVA-merged ★     │ 35.6 │ 87.2      │ 72.5      │ 88.1   │             │
│  └────────────────────┴──────┴───────────┴───────────┴────────┘             │
│                                                                              │
│  Output: → LLaVA-v1.5-7B-merged (safe + helpful + capable)                 │
└──────────────────────────────────────────────────────────────────────────────┘
```

### End-to-End Example 2: OCR-Specialized VLM Pipeline

```
┌──────────────────────────────────────────────────────────────────────────────┐
│              OCR-Specialized VLM — Full 3-Stage Pipeline                      │
├──────────────────────────────────────────────────────────────────────────────┤
│                                                                              │
│  Stage 1: SFT + Knowledge Distillation                                       │
│  ═════════════════════════════════════════                                    │
│  Dataset:  43M pages (PDFs, receipts, invoices, handwriting, tables, etc.)   │
│  Teacher:  Qwen3-VL-235B (frozen, knowledge distillation)                   │
│  Student:  Qwen2.5-VL-7B (trainable except ViT)                            │
│  Objective:                                                                  │
│    L = (1-α)·CE(student, y*) + α·T²·KL(softmax(z_S/T) ‖ softmax(z_T/T))  │
│    α = 0.5, T = 4                                                           │
│  Config:   AdamW (β₁=0.9, β₂=0.95), lr=2e-5, cosine, bf16, batch=384     │
│  Duration: ~7 days on 64×A100                                               │
│                                                                              │
│  ┌───────────────┐    ┌──────────┐    ┌─────────────┐                       │
│  │ Qwen3-VL-235B │    │ Student  │    │ Ground Truth│                       │
│  │ (Teacher)     │    │ 7B model │    │ labels y*   │                       │
│  └──────┬────────┘    └────┬─────┘    └──────┬──────┘                       │
│         │ soft targets     │ z_S             │ y*                            │
│         └────────┐  ┌─────┘                  │                               │
│                  ▼  ▼                        ▼                               │
│            L = α·T²·KL(S‖T) + (1-α)·CE(S, y*)                              │
│                                                                              │
│  Output: → OCR-VLM-base, OCR-VLM-bbox-base                                  │
│                  │                                                            │
│                  ▼                                                            │
│  Stage 2: RLVR via GRPO                                                      │
│  ═══════════════════════                                                      │
│  Algorithm:  GRPO (Group Relative Policy Optimization)                       │
│  Config:     β=0.01, lr=1e-6, ε=0.2, bf16                                  │
│  Tooling:    HuggingFace TRL + vLLM                                          │
│                                                                              │
│  ┌───────────────────────────┐    ┌─────────────────────────────┐           │
│  │  OCR-focused RLVR         │    │  BBox-focused RLVR          │           │
│  │  G = 28 rollouts/prompt   │    │  G = 14 rollouts/prompt     │           │
│  │                           │    │                             │           │
│  │  Reward:                  │    │  Reward:                    │           │
│  │  R = 0.5·(1-CER)         │    │  R = 0.4·mean_IoU          │           │
│  │    + 0.3·format_score    │    │    + 0.3·ID_overlap        │           │
│  │    + 0.2·anti_rep        │    │    + 0.2·count_accuracy    │           │
│  │                           │    │    + 0.1·anti_rep         │           │
│  │  Duration: ~3 days 8×A100│    │  Duration: ~2 days 8×A100  │           │
│  └─────────────┬─────────────┘    └─────────────┬───────────────┘           │
│                │                                │                            │
│                ▼                                ▼                            │
│  → OCR-VLM-rl (CER: 2.1%)       → OCR-VLM-bbox-rl (mAP: 89.1%)           │
│     τ_ocr = θ_rl - θ_base           τ_bbox = θ_rl - θ_base                │
│                │                                │                            │
│                └───────────────┬─────────────────┘                           │
│                                ▼                                             │
│  Stage 3: Model Souping                                                      │
│  ═══════════════════════                                                      │
│  Phase A: Intra-task soup (avg last 3 checkpoints per RL run)               │
│  Phase B: Cross-task merge (task arithmetic)                                 │
│           θ = θ_base + 0.8·τ̃_ocr + 0.6·τ̃_bbox                            │
│                                                                              │
│  ┌────────────────────┬─────────┬─────────┬─────────┬────────┐             │
│  │ Model              │ OCR CER↓│ BBox mAP│ Table F1│ Size   │             │
│  ├────────────────────┼─────────┼─────────┼─────────┼────────┤             │
│  │ SFT-only (KD)      │ 3.8%    │ 82.3%   │ 71.2%   │ 7B     │             │
│  │ GRPO OCR-only      │ 2.1%    │ 79.1%   │ 68.5%   │ 7B     │             │
│  │ GRPO BBox-only     │ 4.2%    │ 89.1%   │ 73.8%   │ 7B     │             │
│  │ Merged (soup) ★    │ 2.4%    │ 87.5%   │ 74.1%   │ 7B     │             │
│  │ Teacher (235B)     │ 1.2%    │ 93.4%   │ 82.6%   │ 235B   │             │
│  └────────────────────┴─────────┴─────────┴─────────┴────────┘             │
│                                                                              │
│  The merged 7B achieves ~85% of the 235B teacher at 1/30th inference cost  │
│  and the merge was FREE (no training, just weight addition).                │
│                                                                              │
│  Output: → OCR-VLM-soup (balanced), OCR-VLM-bbox-soup (bbox-prioritized)   │
└──────────────────────────────────────────────────────────────────────────────┘
```

### When to Use Each Stage — Decision Guide

| Stage | Cost | Duration | What It Fixes | Skip If... |
|-------|------|----------|--------------|------------|
| **SFT** | High | Days–weeks | Foundational capabilities | Using pretrained base |
| **SFT + KD** | High+ | Days–weeks | + Dark knowledge from teacher | No teacher available |
| **RLHF/DPO** | Medium | Hours–days | Hallucinations, safety, style | Task has verifiable answers |
| **RLVR (GRPO)** | Medium | Hours–days | Task-specific precision | No auto-grading metric |
| **Weight Merging** | Near-zero | Minutes | Multi-task balance | Only one RL variant |

```
  Pipeline decision:

  Have a base model?
       │
       ├── NO  ──► Stage 1: SFT (+ KD if teacher available)
       │
       └── YES ──► What needs improving?
                       │
                       ├── Hallucinations/safety ──► DPO (preference pairs)
                       ├── Task accuracy (OCR, math) ──► GRPO (verifiable rewards)
                       └── Need BOTH ──► DPO + GRPO separately ──► Merge (Stage 3)
```

---

**Key Takeaways:**
1. **SFT is the foundation** — teaches capabilities via next-token prediction
2. **RL sharpens** — optimizes non-differentiable rewards beyond what SFT captures
3. **Merging is free** — combines specialists at zero cost (just vector addition)
4. The **3-stage pattern** (SFT → RL → Merge) appears in virtually every SOTA VLM

---
**Next:** Module 04 — Finetuning for Low Compute (LoRA, QLoRA, Adapters!)